# 16 — MIDI 2026: Immutable Artifact Integrity Audit (v1)

**Amaç:** MIDI 2026 camera-ready revizyonundan **önce**, mevcut Stage 08–15 çıktılarının sürüm, SHA-256, satır/anahtar, train/test ayrımı ve Stage 14 metrik bütünlüğünü kontrol etmek.

- **Yeni LLM inference, model eğitimi, hiperparametre araması, CV veya bootstrap yoktur.**
- Eski notebook ve dosyalara **hiçbir yazma işlemi yapılmaz**.
- Yalnızca `results/midi2026_revision/v1/00_integrity_audit/` ve `.../manifests/` klasörlerine yazılır.
- Sabit veri bölmeleri: `validated_v007` (1.000 soru; 30 hariç → 970 Clean-Dev) ve `validated_v012_final250_v1` (1.250 soruluk birleşik matris; 250 bağımsız test).
- Dondurulmuş Stage 14 tahminleri salt okunur olarak değerlendirilir.

**Çalıştırma:** Colab üzerinde varsayılan `INPUT_MODE="drive"` kullanılır. Tüm kaynak dosyalar orijinal Google Drive klasörlerinde olmalıdır. İstersen yalnız yüklenmiş dokuz dosyayla ön inceleme için `INPUT_MODE="local"` kullanabilirsin; bu durumda Stage 12/13 ve final şema gibi bulunmayan kanıtlar **eksik** raporlanır ve tam audit `PASS` sayılmaz.

In [ ]:
# 1 — Ortam (model çalıştırmaz)
import sys, subprocess, importlib.util
if importlib.util.find_spec('pyarrow') is None and importlib.util.find_spec('fastparquet') is None:
    print('Parquet okuyucusu bulunamadı; pyarrow yükleniyor (sadece bağımlılık kurulumu).')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow>=15,<25'])

from pathlib import Path
import json, hashlib, math
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

INPUT_MODE = 'drive'  # 'drive' veya 'local'; Colab Drive kullanımı önerilir
LOCAL_INPUT_DIR = Path('/content')

if INPUT_MODE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/LLM_Confidence_Thesis')
    assert ROOT.is_dir(), f'Proje dizini bulunamadı: {ROOT}'
elif INPUT_MODE == 'local':
    ROOT = LOCAL_INPUT_DIR
    assert ROOT.is_dir(), ROOT
else:
    raise ValueError('INPUT_MODE sadece drive/local olabilir')

OUT_ROOT = ROOT / 'results' / 'midi2026_revision' / 'v1'
OUT_AUDIT = OUT_ROOT / '00_integrity_audit'
OUT_MANIFEST = OUT_ROOT / 'manifests'
DEV_VERSION = 'validated_v007'
TEST_VERSION = 'validated_v012_final250_v1'
CORE = ['mean_logprob','min_logprob','mean_token_entropy_normalized',
        'output_content_token_count','self_confidence_scaled',
        'verification_p_correct','self_consistency','unknown_response']
TARGET = 'manual_is_correct'
KEY = ['model_id','question_id']
print('INPUT_MODE:', INPUT_MODE)
print('SOURCE ROOT:', ROOT)
print('WRITE ROOT (yalnız yeni çıktılar):', OUT_ROOT)

In [ ]:
# 2 — Salt okunur dosya haritası ve kontrol yardımcıları
if INPUT_MODE == 'drive':
    P = {
      'dev_extended': ROOT/'features'/'assembled'/DEV_VERSION/'extended_model_matrix_v2_repair1.parquet',
      'dev_schema': ROOT/'features'/'assembled'/DEV_VERSION/'feature_schema_v1.json',
      'dev_manifest': ROOT/'features'/'assembled'/DEV_VERSION/'stage08_manifest_v2_repair1.json',
      'test_core': ROOT/'features'/'assembled'/TEST_VERSION/'core_model_matrix_v1.parquet',
      'test_extended': ROOT/'features'/'assembled'/TEST_VERSION/'extended_model_matrix_v1.parquet',
      'test_schema': ROOT/'features'/'assembled'/TEST_VERSION/'feature_schema_v1.json',
      'test_manifest': ROOT/'features'/'assembled'/TEST_VERSION/'stage08_manifest_v1.json',
      'frozen_predictions': ROOT/'results'/'stage14'/TEST_VERSION/'final_test_predictions_v1.parquet',
      'predictions_manifest': ROOT/'results'/'stage14'/TEST_VERSION/'stage14_predictions_manifest_v1.json',
      'published_metrics': ROOT/'results'/'stage14'/TEST_VERSION/'final_test_metrics_v1.csv',
    }
else:
    names = {'dev_extended':'extended_model_matrix_v2_repair1.parquet',
     'dev_schema':'feature_schema_v1.json','dev_manifest':'stage08_manifest_v2_repair1.json',
     'test_core':'core_model_matrix_v1.parquet','test_extended':'extended_model_matrix_v1.parquet',
     'test_schema':'__NOT_UPLOADED__final_feature_schema_v1.json',
     'test_manifest':'stage08_manifest_v1.json',
     'frozen_predictions':'final_test_predictions_v1.parquet',
     'predictions_manifest':'stage14_predictions_manifest_v1.json',
     'published_metrics':'final_test_metrics_v1.csv'}
    P = {k:ROOT/v for k,v in names.items()}

checks=[]
def record(name, ok, details='', severity='required'):
    # statuses: PASS, FAIL, MISSING
    status = 'MISSING' if ok is None else ('PASS' if bool(ok) else 'FAIL')
    checks.append({'check':name,'status':status,'severity':severity,'details':str(details)})
    print(f'[{status:7}] {name}: {str(details)[:180]}')

def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()

def readjson(path): return json.loads(Path(path).read_text(encoding='utf-8'))
def canonical(obj): return json.dumps(obj,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def canonical_hash(obj): return hashlib.sha256(canonical(obj).encode('utf-8')).hexdigest()

def sha_check(name,path,expected,required=True):
    if not Path(path).is_file():
        record(name,None,f'Bulunamadı: {path}', 'required' if required else 'optional')
        return False
    actual=digest(path)
    record(name,actual==expected,f'actual={actual}; expected={expected}',
           'required' if required else 'optional')
    return actual==expected

for k,path in P.items():
    record('file_exists:'+k, True if path.is_file() else None,str(path),
           'optional' if k=='test_schema' else 'required')
required_keys=[k for k in P if k!='test_schema']
missing=[k for k in required_keys if not P[k].is_file()]
if missing:raise FileNotFoundError('Gerekli dosyalar eksik: '+', '.join(missing))

In [ ]:
# 3 — SHA-256 ve manifest sözleşmeleri
mdev=readjson(P['dev_manifest']); mtest=readjson(P['test_manifest'])
pm=readjson(P['predictions_manifest']); sd=readjson(P['dev_schema'])

hash_contract={
    'dev_extended':mdev['outputs']['extended_model_matrix_sha256'],
    'dev_schema':mdev['outputs']['feature_schema_file_sha256'],
    'test_core':mtest['outputs']['core_model_matrix_sha256'],
    'test_extended':mtest['outputs']['extended_model_matrix_sha256'],
    'frozen_predictions':pm['predictions_sha256'],
}
for k, expected in hash_contract.items():sha_check('sha256:'+k,P[k],expected)
if P['test_schema'].is_file():
    sha_check('sha256:test_schema',P['test_schema'],mtest['outputs']['feature_schema_file_sha256'])
    st=readjson(P['test_schema'])
else:st=None

record('dataset_version:dev',mdev.get('dataset_version')==DEV_VERSION,mdev.get('dataset_version'))
record('dataset_version:test',mtest.get('dataset_version')==TEST_VERSION,mtest.get('dataset_version'))
record('dataset_version:prediction',pm.get('dataset_version')==TEST_VERSION,pm.get('dataset_version'))
record('shape_contract:manifest_dev',mdev.get('n_questions')==1000 and mdev.get('n_model_answer_rows')==3000,str((mdev.get('n_questions'),mdev.get('n_model_answer_rows'))))
record('shape_contract:manifest_test',mtest.get('n_questions')==1250 and mtest.get('n_model_answer_rows')==3750,str((mtest.get('n_questions'),mtest.get('n_model_answer_rows'))))
record('core_features:dev_manifest',mdev.get('core_features')==CORE,str(mdev.get('core_features')))
record('core_features:test_manifest',mtest.get('core_features')==CORE,str(mtest.get('core_features')))
record('core_features:pred_manifest',pm.get('core_features_ordered')==CORE,str(pm.get('core_features_ordered')))
record('core_features:dev_schema',sd.get('core_features')==CORE,str(sd.get('core_features')))
record('schema:dev_internal_hash',canonical_hash({k:v for k,v in sd.items() if k!='schema_sha256'})==sd.get('schema_sha256'),sd.get('schema_sha256'))
record('schema:dev_manifest_hash',sd.get('schema_sha256')==mdev.get('feature_schema_sha256'),mdev.get('feature_schema_sha256'))
record('target:dev_schema',sd.get('target')==TARGET,sd.get('target'))
if st is not None:
    record('core_features:test_schema',st.get('core_features')==CORE,str(st.get('core_features')))
    record('schema:test_internal_hash',canonical_hash({k:v for k,v in st.items() if k!='schema_sha256'})==st.get('schema_sha256'),st.get('schema_sha256'))
    record('schema:test_manifest_hash',st.get('schema_sha256')==mtest.get('feature_schema_sha256'),mtest.get('feature_schema_sha256'))
    record('target:test_schema',st.get('target')==TARGET,st.get('target'))
record('frozen_predictions:contract',
       pm.get('n_independent_test_questions')==250 and pm.get('n_model_answer_rows')==750 and
       pm.get('n_method_prediction_rows')==3000 and pm.get('training_n_per_model')==970,
       str({k:pm.get(k) for k in ('n_independent_test_questions','n_model_answer_rows','n_method_prediction_rows','training_n_per_model')}))
record('freeze_status',pm.get('status')=='PREDICTIONS_FROZEN_BEFORE_TEST_LABEL_EVALUATION' and pm.get('test_labels_used_in_fit_selection_or_prediction') is False,pm.get('status'))

In [ ]:
# 4 — Parquet satırları, sabit anahtarlar, 970+250 ayrımı (salt okunur)
DEV_COLS=['question_id','model_id','question_group_id','exclude_from_final_test',
          'used_in_model_selection',TARGET]+CORE
TEST_COLS=['question_id','model_id','question_group_id',TARGET]+CORE
D=pd.read_parquet(P['dev_extended'],columns=DEV_COLS)
T=pd.read_parquet(P['test_core'],columns=TEST_COLS)
TE=pd.read_parquet(P['test_extended'],columns=KEY)
F=pd.read_parquet(P['frozen_predictions'])
record('parquet_rows:dev',len(D)==3000,len(D))
record('parquet_rows:test_core',len(T)==3750,len(T))
record('parquet_rows:test_extended',len(TE)==3750,len(TE))
record('parquet_rows:frozen',len(F)==3000,len(F))
record('unique_key:dev',not D.duplicated(KEY).any(),'question_id + model_id')
record('unique_key:test_core',not T.duplicated(KEY).any(),'question_id + model_id')
record('unique_key:test_extended',not TE.duplicated(KEY).any(),'question_id + model_id')
record('unique_key:frozen',not F.duplicated(KEY+['method']).any(),'question_id + model_id + method')
record('key_match:test_core_extended',set(map(tuple,T[KEY].to_numpy()))==set(map(tuple,TE[KEY].to_numpy())),'CORE ↔ EXTENDED')

old_ids=set(D['question_id']); test_full_ids=set(T['question_id'])
methods=set(pm['methods']); required_methods={'raw_verification','train_prevalence','logistic_core','xgb_core'}
record('methods:frozen',methods==required_methods,str(sorted(methods)))
record('method_rows:frozen',set(F['method'])==required_methods and F.groupby('method').size().eq(750).all(),str(F.groupby('method').size().to_dict()))
selected_ids=set(F['question_id'])
record('questions:dev_1000',len(old_ids)==1000,len(old_ids))
record('questions:test_matrix_1250',len(test_full_ids)==1250,len(test_full_ids))
record('questions:selected_250',len(selected_ids)==250,len(selected_ids))
record('questions:dev_test_disjoint',old_ids.isdisjoint(selected_ids),f'intersection={len(old_ids & selected_ids)}')
record('questions:test_matrix_union',test_full_ids==(old_ids | selected_ids),f'missing={len((old_ids | selected_ids)-test_full_ids)}, extra={len(test_full_ids-(old_ids|selected_ids))}')

model_ids=set(D['model_id'])
record('model_ids:all_same',len(model_ids)==3 and model_ids==set(T['model_id'])==set(F['model_id']),str(sorted(model_ids)))
record('questions:per_model',
       D.groupby('model_id').question_id.nunique().eq(1000).all() and
       T.groupby('model_id').question_id.nunique().eq(1250).all() and
       F.groupby(['model_id','method']).question_id.nunique().eq(250).all(),
       'dev=1000; combined=1250; frozen=250/model-method')

C=D.loc[D['exclude_from_final_test'].astype(int).eq(0)].copy()
record('train:970_each',len(C)==2910 and C.groupby('model_id').size().eq(970).all(),str(C.groupby('model_id').size().to_dict()))
record('train:no_model_selection_overlap',not C['used_in_model_selection'].astype(bool).any(),'Clean-Dev excludes Stage03')
record('train:30_excluded_each',D.loc[D['exclude_from_final_test'].astype(int).ne(0)].groupby('model_id').size().eq(30).all(),str(D.loc[D['exclude_from_final_test'].astype(int).ne(0)].groupby('model_id').size().to_dict()))
record('labels:development',C[TARGET].notna().all() and C[TARGET].isin([0,1]).all(),'0/1 & non-null')
T250=T.loc[T['question_id'].isin(selected_ids)].copy()
record('labels:final250',len(T250)==750 and T250[TARGET].notna().all() and T250[TARGET].isin([0,1]).all(),'750 labels, 0/1 & non-null')

for label,frame in [('clean_dev',C),('test_selected',T250)]:
    a=frame[CORE].apply(pd.to_numeric,errors='coerce').to_numpy(dtype=float)
    record('predictors:finite_'+label,np.isfinite(a).all(),f'{a.shape}, NaN/inf={int((~np.isfinite(a)).sum())}')
    for field in ['self_confidence_scaled','verification_p_correct','self_consistency','unknown_response']:
        x=pd.to_numeric(frame[field],errors='coerce')
        record('predictor_range:'+label+':'+field, x.between(0,1).all(),f'n_out={int((~x.between(0,1)).sum())}')

record('probabilities:frozen',F['p_correct'].notna().all() and pd.to_numeric(F['p_correct'],errors='coerce').between(0,1).all(), 'p_correct in [0,1]')
record('frozen:selected_pairs',set(map(tuple,F[KEY].drop_duplicates().to_numpy()))==set(map(tuple,T250[KEY].to_numpy())),'frozen predictions match selected test responses')
record('frozen:raw_verification_matches_feature',
       np.allclose(F.loc[F['method'].eq('raw_verification'),KEY+['p_correct']].merge(T250[KEY+['verification_p_correct']],on=KEY,validate='one_to_one')['p_correct'].to_numpy(),
                   F.loc[F['method'].eq('raw_verification'),KEY+['p_correct']].merge(T250[KEY+['verification_p_correct']],on=KEY,validate='one_to_one')['verification_p_correct'].to_numpy(),rtol=0,atol=1e-12),
       'raw predictions identical to frozen verification feature')

In [ ]:
# 5 — Stage 14 raporunu dondurulmuş tahminlerden yeniden HESAPLAMADAN değil,
# yalnız metrik formülleriyle bağımsız doğrula. Eğitim/inference/bootstrapping yoktur.
# Bu aşama final test etiketlerini audit için okur: Stage 14 zaten tamamlanmış durumda.
published=pd.read_csv(P['published_metrics'])
L=T250[KEY+[TARGET]].copy()
E=F.merge(L,on=KEY,how='left',validate='many_to_one',indicator=True)
record('frozen_labels:join',len(E)==3000 and E['_merge'].eq('both').all(),f'rows={len(E)}')

rows=[]
for (model,method),g in E.groupby(['model_id','method'],sort=True):
    g=g.sort_values('question_id',kind='mergesort')
    q=g['question_id'].astype(str).to_numpy()
    y=g[TARGET].to_numpy(dtype=int)
    p=g['p_correct'].to_numpy(dtype=float)
    bins=np.minimum((np.clip(p,0.,1.)*10).astype(int),9)
    ece=sum(float(np.mean(bins==j))*abs(float(np.mean(y[bins==j]))-float(np.mean(p[bins==j])))
            for j in range(10) if (bins==j).any())
    safe=np.clip(p,1e-6,1.-1e-6)
    order=np.lexsort((q,-p))
    aurc=float(np.mean(np.cumsum(1-y[order])/np.arange(1,len(y)+1)))
    rows.append({'model_id':model,'method':method,'n':len(y),'prevalence':float(y.mean()),
                 'brier':float(np.mean((p-y)**2)),
                 'nll':float(-np.mean(y*np.log(safe)+(1-y)*np.log1p(-safe))),
                 'ece':ece,'auroc':float(roc_auc_score(y,p)),
                 'auprc':float(average_precision_score(y,p)), 'aurc':aurc})
recomputed=pd.DataFrame(rows).sort_values(['model_id','method']).reset_index(drop=True)
pp=published.sort_values(['model_id','method']).reset_index(drop=True)
record('metrics:row_keys',set(map(tuple,recomputed[['model_id','method']].to_numpy()))==set(map(tuple,pp[['model_id','method']].to_numpy())),'12 model-method pairs')
metric_cols=['n','prevalence','brier','nll','ece','auroc','auprc','aurc']
max_err={c:float(np.max(np.abs(recomputed[c]-pp[c]))) for c in metric_cols}
record('metrics:stage14_reproduction',len(recomputed)==len(pp)==12 and all(d<=2e-9 for d in max_err.values()), str(max_err))
expected_accuracy={'Qwen/Qwen3-4B':.416,'google/gemma-3-4b-it':.624,'microsoft/Phi-4-mini-instruct':.268}
observed_accuracy=E.loc[E['method'].eq('raw_verification')].groupby('model_id')[TARGET].mean().to_dict()
record('labels:model_test_accuracy',set(observed_accuracy)==set(expected_accuracy) and
       all(abs(float(observed_accuracy[k])-v)<1e-12 for k,v in expected_accuracy.items()),
       str(observed_accuracy))
print(recomputed[['model_id','method','n','prevalence','brier','ece','auroc','aurc']].to_string(index=False))

In [ ]:
# 6 — Stage 14 manifestindeki upstream provenance zinciri (Drive'da tam kontrol)
# Stage12/13 dosyalarının yokluğu model başarısızlığı değil: eksik kanıt olarak kaydedilir.
missing_provenance=[]
for i,entry in enumerate(pm['provenance']):
    expected=entry['sha256']
    original=Path(entry['path'])
    if INPUT_MODE=='drive':
        # Tüm proje köklerini ROOT'a taşı; yalnız orijinal proje dosyaları okunur.
        prefix='/content/drive/MyDrive/LLM_Confidence_Thesis/'
        assert str(original).startswith(prefix),f'Bilinmeyen provenance konumu: {original}'
        source=ROOT/str(original)[len(prefix):]
    else:
        # final ve dev schema aynı ada sahip; yerel modda karıştırma.
        if original.name=='feature_schema_v1.json':
            source=P['test_schema'] if TEST_VERSION in str(original) else P['dev_schema']
        else:
            source=ROOT/original.name
    if source.is_file():
        sha_check(f'provenance:{i:02d}:{source.name}',source,expected)
    else:
        record(f'provenance:{i:02d}:{original.name}',None,f'Eksik kanıt: {original}','required')
        missing_provenance.append(str(original))

# Optional validation: Stage14 evaluation manifest, if present, ties metrics CSV to SHA.
if INPUT_MODE=='drive':
    ev_path=ROOT/'results'/'stage14'/TEST_VERSION/'stage14_evaluation_manifest_v1.json'
else: ev_path=ROOT/'stage14_evaluation_manifest_v1.json'
if ev_path.is_file():
    ev=readjson(ev_path)
    record('stage14_evaluation_manifest:metrics_sha',ev.get('metric_csv_sha256')==digest(P['published_metrics']),str(ev.get('metric_csv_sha256')))
    record('stage14_evaluation_manifest:prediction_sha',ev.get('source_predictions_sha256')==digest(P['frozen_predictions']),str(ev.get('source_predictions_sha256')))
else:
    record('stage14_evaluation_manifest:exists',None,str(ev_path),'optional')

In [ ]:
# 7 — Tek yazma adımı: izlenebilir, idempotent, salt-yeni-klasör raporları
# Daha önceki sürümler / Stage08–15 dizinleri kesinlikle değiştirilmez.
from io import StringIO

status_df=pd.DataFrame(checks)
required_bad=status_df[(status_df['severity']=='required') & status_df['status'].ne('PASS')]
status=('PASS' if required_bad.empty else ('FAILED' if required_bad['status'].eq('FAIL').any() else 'INCOMPLETE'))

input_hashes={k:(digest(v) if v.is_file() else None) for k,v in sorted(P.items())}
summary={
  'audit_version':'midi2026_stage16_v1',
  'status':status,
  'input_mode':INPUT_MODE,
  'dataset_versions':{'development':DEV_VERSION,'test':TEST_VERSION},
  'clean_dev_answers':int(len(C)),
  'clean_dev_questions_per_model':970,
  'frozen_final_test_answers':int(len(T250)),
  'frozen_final_test_questions':int(len(selected_ids)),
  'frozen_prediction_rows':int(len(F)),
  'input_sha256':input_hashes,
  'provenance_missing':missing_provenance,
  'counts_by_status':{str(k):int(v) for k,v in status_df['status'].value_counts().items()},
  'required_unresolved_checks':required_bad['check'].tolist(),
  'notes':['No LLM inference, calibrator fitting or bootstrap was performed.',
           'Data integrity audit uses existing final-test labels after Stage14 evaluation.',
           'New reviewer-requested analyses must be labeled post-review/exploratory.',
           'Final-test schema is a separate file from validated_v007 schema.']
}

report_lines=[
 '# MIDI2026 Stage 16 — Artifact Integrity Audit',
 '',f'**Status:** {status}',
 '',f'- Clean-Dev: {len(C)} model-answer rows (970/model expected)',
 f'- Selected final test: {len(selected_ids)} questions, {len(T250)} model answers',
 f'- Frozen predictions: {len(F)} rows',
 f'- Upstream provenance entries missing: {len(missing_provenance)}',
 '', '## Checks',
 '', '| Check | Result | Severity | Details |', '|---|---|---|---|',
]
for r in checks:
    details=str(r['details']).replace('|','/').replace('\n',' ')[:130]
    report_lines.append(f"| {r['check']} | {r['status']} | {r['severity']} | {details} |")
report_lines+=['','## Interpretation',
 'PASS yalnız tüm zorunlu kontroller geçerse raporlanır.',
 'FAILED veya INCOMPLETE durumunda manifest uyuşmazlıkları ve eksik upstream kanıtları kontrol edilmeden yeni deneylere geçilmemelidir.',
 'Kayıtlar sadece yeni revision/v1 klasörüne yazılmıştır.']

OUT_AUDIT.mkdir(parents=True,exist_ok=True)
OUT_MANIFEST.mkdir(parents=True,exist_ok=True)

def immutable_write(path, blob):
    path=Path(path)
    if path.exists():
        assert path.read_bytes()==blob, (f'IMMUTABILITY CONFLICT: {path}; '
                                          'aynı sürümün içeriğini değiştirme, v2 oluştur')
        print('UNCHANGED:',path)
    else:
        with path.open('xb') as f:f.write(blob)
        print('CREATED:',path)
    return digest(path)

checks_csv=status_df.to_csv(index=False,lineterminator='\n').encode('utf-8')
metrics_csv=recomputed.to_csv(index=False,float_format='%.15g',lineterminator='\n').encode('utf-8')
report_blob=('\n'.join(report_lines)+'\n').encode('utf-8')
summary_blob=(json.dumps(summary,ensure_ascii=False,indent=2,sort_keys=True)+'\n').encode('utf-8')
output_digests={
 'audit_checks_v1.csv':immutable_write(OUT_AUDIT/'audit_checks_v1.csv',checks_csv),
 'stage14_metrics_recomputed_v1.csv':immutable_write(OUT_AUDIT/'stage14_metrics_recomputed_v1.csv',metrics_csv),
 'audit_report_v1.md':immutable_write(OUT_AUDIT/'audit_report_v1.md',report_blob),
 'audit_summary_v1.json':immutable_write(OUT_AUDIT/'audit_summary_v1.json',summary_blob),
}
freeze={
  'stage':'16_midi2026_integrity_audit',
  'audit_version':'v1',
  'status':status,
  'source_hashes':input_hashes,
  'output_sha256':output_digests,
  'required_unresolved_checks':required_bad['check'].tolist(),
  'model_training_performed':False,
  'model_inference_performed':False,
  'source_artifacts_modified':False,
}
immutable_write(OUT_MANIFEST/'stage16_audit_manifest_v1.json',
                (json.dumps(freeze,ensure_ascii=False,sort_keys=True,indent=2)+'\n').encode('utf-8'))
print('\nFINAL STATUS:',status)
if status!='PASS':print('UNRESOLVED REQUIRED CHECKS:',required_bad[['check','status']].to_string(index=False))
print('REPORT:',OUT_AUDIT/'audit_report_v1.md')